In [1]:
# Step 1: Import libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Lambda
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


/home/user/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Step 2: Prepare the dataset
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the quick brown dog jumps over the lazy fox",
    "a man is walking on the street",
    "a dog is barking at the cat",
    "the street is busy with people"
]

In [3]:
# Step 3: Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

print("Vocabulary Size:", vocab_size)

sequences = tokenizer.texts_to_sequences(corpus)


Vocabulary Size: 21


In [4]:
# Step 4: Generate CBOW training data
window_size = 2
contexts = []
targets = []

for sequence in sequences:
    for i, word in enumerate(sequence):
        context = [
            sequence[j]
            for j in range(max(0, i - window_size), min(len(sequence), i + window_size + 1))
            if j != i
        ]

        if context:
            contexts.append(context)
            targets.append(word)

# Pad context sequences
X = pad_sequences(contexts, padding="post")
y = np.array(targets)

print("Training samples:", len(X))

Training samples: 38


In [5]:
# Step 5: Build the CBOW model
embedding_dim = 32
max_len = X.shape[1]

input_context = Input(shape=(max_len,))

embedding = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
)(input_context)

# Average the context word embeddings
average = Lambda(lambda x: tf.reduce_mean(x, axis=1))(embedding)

# Predict the target word
output = Dense(vocab_size, activation="softmax")(average)
cbow_model = Model(input_context, output)

In [6]:
# Step 6: Compile the model
cbow_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cbow_model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 4)]               0         
                                                                 
 embedding (Embedding)       (None, 4, 32)             672       
                                                                 
 lambda (Lambda)             (None, 32)                0         
                                                                 
 dense (Dense)               (None, 21)                693       
                                                                 
Total params: 1,365
Trainable params: 1,365
Non-trainable params: 0
_________________________________________________________________


In [7]:
# Step 6: Compile the model
cbow_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cbow_model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 4)]               0         
                                                                 
 embedding (Embedding)       (None, 4, 32)             672       
                                                                 
 lambda (Lambda)             (None, 32)                0         
                                                                 
 dense (Dense)               (None, 21)                693       
                                                                 
Total params: 1,365
Trainable params: 1,365
Non-trainable params: 0
_________________________________________________________________


In [8]:
# Step 8: Display learned word embeddings
word_embeddings = cbow_model.layers[1].get_weights()[0]

print("\nLearned Word Embeddings:")
for word, index in word_index.items():
    print(word, ":", word_embeddings[index][:5])



Learned Word Embeddings:
the : [-0.04348151  0.0088017   0.00250899  0.04892501  0.02908185]
dog : [ 9.8835677e-05  5.4287799e-03  2.4202261e-02 -5.2295923e-03
  3.5183597e-02]
is : [-0.01819775  0.00893494  0.00906377  0.02473552 -0.03124559]
quick : [ 0.04914166  0.02494745  0.00725415  0.04175379 -0.01822703]
brown : [ 0.01240419 -0.02453491  0.00178628  0.02263251 -0.01126004]
fox : [-0.03379362 -0.0480553  -0.04656496  0.01830905 -0.00169972]
jumps : [ 0.02855971  0.00914528  0.0303924  -0.01730754 -0.01565157]
over : [ 0.02114544  0.04693304 -0.04005406  0.04824622 -0.00186164]
lazy : [ 0.02579344 -0.04383333  0.00433796  0.0071389  -0.03450984]
a : [-0.00614176 -0.03998656 -0.04389603 -0.01642291  0.02064675]
street : [-0.02353069 -0.03096039 -0.02768049  0.00012184  0.03572345]
man : [ 0.04226189  0.01370194 -0.00711347 -0.01160909 -0.04877247]
walking : [ 0.00690398  0.00955021  0.00065919 -0.04517276  0.00980498]
on : [-0.01526288  0.04939909 -0.02523479 -0.0420432  -0.02594

In [9]:

# Step 9: Predict a word from context
def predict_word(context_words):
    context_ids = [
        word_index[word.lower()]
        for word in context_words
        if word.lower() in word_index
    ]

    if not context_ids:
        print("No known context words found.")
        return

    padded_context = pad_sequences(
        [context_ids],
        maxlen=max_len,
        padding="post"
    )

    prediction = cbow_model.predict(padded_context, verbose=0)
    predicted_id = np.argmax(prediction[0])

    for word, index in word_index.items():
        if index == predicted_id:
            print("\nContext words:", context_words)
            print("Predicted target word:", word)
            return

In [10]:
# Example prediction
predict_word(["quick", "brown", "over", "the"])


Context words: ['quick', 'brown', 'over', 'the']
Predicted target word: is
